# PFA - Reconnaissance des Emotions Faciales
## Etape 4b : Deep Learning - CNN Personnalise

**Equipe :** Ayman TEMSAMANI, Nouhaila AIT ABDELAH, Rabab ZITI  
**Encadrant :** Pr. Redouane DAHBI  
**Date :** Semaine S7-S11 (Phase 4 du planning)

**Contexte :** Conformement au cahier des charges (Section 5.2), cette etape implemente un reseau de neurones convolutif (CNN) personnalise. L'objectif est de depasser 85% de precision sur FER2013. Ce modele sera compare au SVM baseline (44%) et aux architectures ResNet50 et VGG16.

---
### Justification des choix architecturaux

**Pourquoi 3-4 couches Conv2D avec filtres croissants (64→128→256) ?**
- La 1ere couche capture les features simples : contours, lignes.
- La 2eme couche combine ces features en motifs : yeux, bouche.
- La 3eme couche apprend des concepts de haut niveau : l'expression entiere.
- Filtres croissants (64→128→256) compensent la reduction spatiale du pooling.

**Pourquoi BatchNormalization ?**
- Stabilise et accelere l'entrainement.
- Reduit le probleme de vanishing gradient.
- Agit comme un regularisateur leger.

**Pourquoi Dropout (0.25, 0.5) ?**
- Empeche le sur-apprentissage en desactivant aleatoirement des neurones.
- 0.25 pour les couches convolutives (moins de parametres).
- 0.5 pour les couches denses (beaucoup de parametres).

**Pourquoi Adam comme optimiseur ?**
- Taux d'apprentissage adaptatif par parametre.
- Convergence plus rapide que SGD.
- Standard pour la classification d'images.

**Pourquoi categorical crossentropy ?**
- Fonction de perte standard pour classification multi-classe (>2 classes).
- Compatible avec l'encodage one-hot des labels.

In [ ]:
# ============================================================
#  SECTION 1 : Import des bibliotheques
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Scikit-learn pour metriques
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

print("Bibliotheques importees avec succes.")
print(f"TensorFlow version : {tf.__version__}")

In [ ]:
# ============================================================
#  SECTION 2 : Chargement des donnees avec data augmentation
# ============================================================

TRAIN_DIR = 'C:/Users/ayman/Desktop/PFA/archive/train'
TEST_DIR  = 'C:/Users/ayman/Desktop/PFA/archive/test'

print("=" * 60)
print("CHARGEMENT AVEC DATA AUGMENTATION")
print("=" * 60)

# Train : AVEC augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Test : SANS augmentation
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(48, 48),
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=64,
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(48, 48),
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=64,
    shuffle=False
)

print(f"\nClasses : {train_generator.class_indices}")
print(f"Train : {train_generator.samples} images, {len(train_generator)} batchs")
print(f"Test  : {test_generator.samples} images, {len(test_generator)} batchs")

In [ ]:
# ============================================================
#  SECTION 3 : Construction du CNN personnalise
# ============================================================

def build_cnn(input_shape=(48, 48, 1), num_classes=7):
    """
    CNN personnalise conforme au cahier des charges (Section 5.2) :
    - 3 couches Conv2D avec filtres croissants (64, 128, 256)
    - BatchNormalization apres chaque Conv2D
    - MaxPooling2D pour reduction spatiale
    - Dropout pour anti-overfitting
    - 2 couches Dense finales
    """
    model = Sequential([
        # Bloc 1 : 64 filtres 3x3
        Conv2D(64, (3,3), padding='same', activation='relu', input_shape=input_shape),
        BatchNormalization(),
        Conv2D(64, (3,3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2,2)),
        Dropout(0.25),
        
        # Bloc 2 : 128 filtres 3x3
        Conv2D(128, (3,3), padding='same', activation='relu'),
        BatchNormalization(),
        Conv2D(128, (3,3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2,2)),
        Dropout(0.25),
        
        # Bloc 3 : 256 filtres 3x3
        Conv2D(256, (3,3), padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2,2)),
        Dropout(0.25),
        
        # Couches denses
        Flatten(),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        
        # Sortie : 7 classes
        Dense(num_classes, activation='softmax')
    ])
    
    return model

# Creation du modele.
model = build_cnn()

print("=" * 60)
print("ARCHITECTURE DU CNN")
print("=" * 60)
model.summary()

In [ ]:
# ============================================================
#  SECTION 4 : Compilation du modele
# ============================================================

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Modele compile avec :")
print("  - Optimizer : Adam (lr=0.001)")
print("  - Loss      : categorical_crossentropy")
print("  - Metrics   : accuracy")

In [ ]:
# ============================================================
#  SECTION 5 : Callbacks (optimisation de l'entrainement)
# ============================================================

# ReduceLROnPlateau : reduit le learning rate si pas d'amelioration.
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# EarlyStopping : arrete l'entrainement si plus d'amelioration.
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

# ModelCheckpoint : sauvegarde le meilleur modele.
checkpoint = ModelCheckpoint(
    'best_cnn_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks = [reduce_lr, early_stop, checkpoint]

print("Callbacks configures :")
print("  - ReduceLROnPlateau (patience=5, factor=0.5)")
print("  - EarlyStopping (patience=15)")
print("  - ModelCheckpoint (best model saved)")

In [ ]:
# ============================================================
#  SECTION 6 : Entrainement du CNN
# ============================================================

print("=" * 60)
print("ENTRAINEMENT DU CNN")
print("Cette operation peut prendre 1-3 heures...")
print("=" * 60)

EPOCHS = 80

start_time = time.time()

history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=EPOCHS,
    validation_data=test_generator,
    validation_steps=len(test_generator),
    callbacks=callbacks,
    verbose=1
)

training_time = time.time() - start_time
print(f"\nEntrainement termine en {training_time/60:.1f} minutes.")

In [ ]:
# ============================================================
#  SECTION 7 : Courbes d'apprentissage
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy.
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy du CNN', fontsize=14)
axes[0].set_xlabel('Epoque')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss.
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss du CNN', fontsize=14)
axes[1].set_xlabel('Epoque')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  SECTION 8 : Evaluation sur le jeu de test
# ============================================================

print("=" * 60)
print("EVALUATION DU CNN SUR LE JEU DE TEST")
print("=" * 60)

# Prediction.
test_generator.reset()
y_pred_probs = model.predict(test_generator, steps=len(test_generator))
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

# Metriques.
accuracy = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')

emotion_names = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

print(f"\n{'='*40}")
print(f"Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1-Score (macro) : {f1_macro:.4f}")
print(f"{'='*40}")

print("\nRapport de classification :")
print(classification_report(y_true, y_pred, target_names=emotion_names, zero_division=0))

In [ ]:
# ============================================================
#  SECTION 9 : Matrice de confusion
# ============================================================

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=emotion_names, yticklabels=emotion_names)
plt.title('Matrice de Confusion - CNN Custom', fontsize=16)
plt.xlabel('Emotion predite')
plt.ylabel('Emotion reelle')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  SECTION 10 : Temps d'inference
# ============================================================

print("=" * 60)
print("TEMPS D'INFERENCE")
print("=" * 60)

# Prendre un batch.
test_generator.reset()
batch = next(test_generator)

# Mesurer le temps d'inference.
n_runs = 100
start = time.time()
for _ in range(n_runs):
    _ = model.predict(batch[0], verbose=0)
elapsed = (time.time() - start) / n_runs * 1000

print(f"\nTemps d'inference par batch de 64 : {elapsed:.2f} ms")
print(f"Temps d'inference par image       : {elapsed/64:.2f} ms")
print(f"FPS equivalent                    : {1000/(elapsed/64):.1f}")
print(f"\nObjectif cahier des charges : > 15 FPS")

In [ ]:
# ============================================================
#  SECTION 11 : Sauvegarde du modele
# ============================================================

model.save('cnn_emotion_model.h5')
print("Modele sauvegarde : cnn_emotion_model.h5")
print("\nLe meilleur modele (checkpoint) : best_cnn_model.h5")

In [ ]:
# ============================================================
#  SECTION 12 : Comparaison SVM vs CNN
# ============================================================

print(f"""
============================================================
COMPARAISON SVM (baseline) vs CNN
============================================================

| Modele        | Accuracy | F1-Score | Temps inference |
|---------------|----------|----------|-----------------|
| SVM + HOG     | 44.24%   | 0.4528   | ~0.1 ms         |
| CNN Custom    | {accuracy*100:.2f}%    | {f1_macro:.4f}  | {elapsed/64:.2f} ms       |

Amelioration : +{(accuracy-0.4424)*100:.1f}% de precision

Objectif cahier des charges : > 85%
============================================================
""")

In [ ]:
# ============================================================
#  SECTION 13 : Bilan de l'etape 4b
# ============================================================

print("\n" + "=" * 60)
print("BILAN DE L'ETAPE 4b - CNN CUSTOM")
print("=" * 60)

print(f"""
Architecture :
  - 6 couches Conv2D (64→64→128→128→256)
  - BatchNormalization apres chaque Conv2D
  - 3 MaxPooling2D (2x2)
  - Dropout (0.25 conv, 0.5 dense)
  - 2 couches Dense (512, 256)
  - Softmax 7 classes

Parametres d'entrainement :
  - Optimizer : Adam (lr=0.001)
  - Loss : categorical_crossentropy
  - Batch size : 64
  - Callbacks : ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

Resultats :
  - Accuracy : {accuracy*100:.2f}%
  - F1-Score macro : {f1_macro:.4f}
  - Temps inference : {elapsed/64:.2f} ms/image

Prochaines etapes :
  -> Etape 4c : ResNet50 (transfer learning)
  -> Etape 4d : VGG16 (transfer learning)
  -> Comparaison finale des 3 architectures
""")